# Morgan Stanley India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** morganstanley.eightfold.ai/careers

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:57:36
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Morgan_Stanley"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Morgan_Stanley/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("MORGAN STANLEY INDIA JOB SCRAPER (Fixed v2)")
print("ATS: Eightfold AI (morganstanley.eightfold.ai)")
print("=" * 60)

import requests, time, random
from bs4 import BeautifulSoup

morgan_stanley_jobs = []
domain = "morganstanley.eightfold.ai"
session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Referer": f"https://{domain}/careers",
})

# Eightfold v2 API — try multiple endpoint and param patterns
ENDPOINTS = [
    f"https://{domain}/api/apply/v2/jobs",
    f"https://{domain}/api/jobs/search",
]

INDIA_KEYWORDS = ["india", "bengaluru", "bangalore", "hyderabad", "mumbai",
                  "pune", "chennai", "delhi", "gurugram", "noida"]

page_size = 50
found = False

for api_url in ENDPOINTS:
    if found:
        break
    print(f"  Trying endpoint: {api_url}")

    # Try both GET with params and POST with body
    for attempt_type in ["get_location", "get_all"]:
        page = 1
        attempt_jobs = []

        while len(attempt_jobs) < 1000:
            if attempt_type == "get_location":
                params = {
                    "num": page_size,
                    "start": (page - 1) * page_size,
                    "location": "India",
                    "domain": "morganstanley",
                }
            else:
                params = {
                    "num": page_size,
                    "start": (page - 1) * page_size,
                    "domain": "morganstanley",
                }

            try:
                resp = session.get(api_url, params=params, timeout=30)
                if resp.status_code != 200:
                    print(f"  [{attempt_type}] HTTP {resp.status_code}")
                    break

                data = resp.json()
                positions = data.get("positions", data.get("jobs", data.get("results", [])))

                if not positions:
                    break

                print(f"  [{attempt_type}] Page {page}: {len(positions)} positions")

                for pos in positions:
                    loc = (pos.get("location", "") or
                           pos.get("city", "") or
                           pos.get("locations", [""])[0] if isinstance(pos.get("locations"), list) else "")
                    loc_str = str(loc).lower()

                    # Include if location matches India keywords, or if we're fetching all
                    if attempt_type == "get_all" and not any(k in loc_str for k in INDIA_KEYWORDS):
                        continue

                    city = str(loc).split(",")[0].strip() if loc else "India"
                    jd_text = html_to_text(pos.get("description", pos.get("summary", "")))
                    job_id = str(pos.get("id", pos.get("requisition_id", pos.get("job_id", ""))))
                    job_url = pos.get("apply_url", f"https://{domain}/careers?pid={job_id}" if job_id else "")
                    dept = pos.get("team", pos.get("department", pos.get("category", "")))

                    attempt_jobs.append({
                        "job_id": job_id,
                        "title": pos.get("name", pos.get("title", "")),
                        "company_name": "Morgan Stanley",
                        "job_url": job_url,
                        "business_unit": str(dept) if dept else "",
                        "raw_jd_text": jd_text,
                        "location_city": city,
                        "location_country": "India",
                        "industry": "Financial Services / Investment Banking",
                        "date_posted": str(pos.get("t_update", pos.get("updated_at",
                                          datetime.now().strftime("%Y-%m-%d"))))[:10],
                        "is_active": True,
                        "salary_currency": "INR",
                        "source_platform": "Eightfold (Morgan Stanley)",
                    })

                if len(positions) < page_size:
                    break
                page += 1
                time.sleep(random.uniform(0.5, 1.5))

            except Exception as e:
                print(f"  [{attempt_type}] Error: {e}")
                break

        if attempt_jobs:
            print(f"  SUCCESS with [{attempt_type}]: {len(attempt_jobs)} India jobs")
            morgan_stanley_jobs = attempt_jobs
            found = True
            break

# Selenium fallback if API fails
if not morgan_stanley_jobs:
    print("\n  API failed — trying Selenium fallback on morganstanley.eightfold.ai...")
    driver = setup_selenium()
    try:
        driver.get(f"https://{domain}/careers?location=India")
        time.sleep(10)
        soup = BeautifulSoup(driver.page_source, "lxml")
        cards = soup.select("[class*='job-card'], [class*='position'], [data-job-id], a[href*='/careers?pid']")
        for card in cards:
            title_el = card.select_one("h3, h2, [class*='title'], a")
            title = title_el.get_text(strip=True) if title_el else ""
            href = card.get("href", card.select_one("a[href]").get("href", "") if card.select_one("a[href]") else "")
            if is_valid_job_title(title):
                morgan_stanley_jobs.append({
                    "job_id": href.split("pid=")[-1] if "pid=" in href else str(len(morgan_stanley_jobs)),
                    "title": title,
                    "company_name": "Morgan Stanley",
                    "job_url": href if href.startswith("http") else f"https://{domain}{href}",
                    "business_unit": "",
                    "raw_jd_text": card.get_text(" ", strip=True),
                    "location_city": "India",
                    "location_country": "India",
                    "industry": "Financial Services / Investment Banking",
                    "date_posted": datetime.now().strftime("%Y-%m-%d"),
                    "is_active": True,
                    "salary_currency": "INR",
                    "source_platform": "Eightfold Selenium (Morgan Stanley)",
                })
    except Exception as e:
        print(f"  Selenium error: {e}")
    finally:
        driver.quit()

print(f"Total Morgan Stanley India jobs: {len(morgan_stanley_jobs)}")


MORGAN STANLEY INDIA JOB SCRAPER (Fixed v2)
ATS: Eightfold AI (morganstanley.eightfold.ai)
  Trying endpoint: https://morganstanley.eightfold.ai/api/apply/v2/jobs


  [get_location] HTTP 404


  [get_all] HTTP 404
  Trying endpoint: https://morganstanley.eightfold.ai/api/jobs/search


  [get_location] HTTP 404


  [get_all] HTTP 404

  API failed — trying Selenium fallback on morganstanley.eightfold.ai...


Total Morgan Stanley India jobs: 1


In [5]:
df_morgan_stanley = save_results(morgan_stanley_jobs, "Morgan Stanley", OUTPUT_DIR)
if df_morgan_stanley is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_morgan_stanley.columns]
    print(df_morgan_stanley[cols].head(10).to_string())


  [OK] Saved 1 jobs -> Morgan Stanley_jobs_2026-03-31.csv
       Seniority: {'senior': 1}
       Work mode: {'onsite': 1}
       Has JD text: 1/1
       Has job URL: 1/1
       Has business unit: 0/1

Sample jobs:
                                                                           title location_city seniority_level business_unit                             job_url
0  Portfolio Analytics Quants , ISG Operations, Senior Associate , Fund Services         India          senior                https://morganstanley.eightfold.ai
